# Smoke Test

Verifies: clean imports, 10-second generation, 10-step LoRA benchmark.

**Run this before any other notebook.**

In [ ]:
# Cell 1: Install pinned dependencies
import subprocess
subprocess.run('pip install -q transformers==4.57.6 huggingface_hub==0.36.0 '
               'accelerate==1.12.0 "peft==0.18.*"', shell=True, check=True)
subprocess.run('pip install -q --no-deps '
               'git+https://github.com/jthickstun/anticipation.git', shell=True, check=True)
subprocess.run('pip install -q mido==1.2.10 pretty_midi scipy scikit-learn', shell=True, check=True)
print('Installs done.')

In [ ]:
# Cell 2: Verify imports and record library versions
import torch, transformers, peft, anticipation
print(f'torch       {torch.__version__}')
print(f'transformers {transformers.__version__}')
print(f'peft        {peft.__version__}')
print(f'cuda        {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU         {torch.cuda.get_device_name(0)}')

In [ ]:
# Cell 3: Inspect anticipation config — record CONTROL_OFFSET and vocab size
import anticipation.config as acfg
print('CONTROL_OFFSET =', acfg.CONTROL_OFFSET)  # UNVERIFIED until run
print('anticipation config attrs:', [a for a in dir(acfg) if not a.startswith('_')])

In [ ]:
# Cell 4: Load model in fp16
# Try AutoModelForCausalLM first; fall back to GPT2LMHeadModel if architectures field fails.
import torch
from transformers import AutoTokenizer

MODEL_NAME = 'stanford-crfm/music-medium-800k'

try:
    from transformers import AutoModelForCausalLM
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16)
    print('Loaded with AutoModelForCausalLM')
except Exception as e:
    print(f'AutoModelForCausalLM failed ({e}), trying GPT2LMHeadModel')
    from transformers import GPT2LMHeadModel
    model = GPT2LMHeadModel.from_pretrained(MODEL_NAME, torch_dtype=torch.float16)
    print('Loaded with GPT2LMHeadModel')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
model.eval()
print(f'vocab_size = {model.config.vocab_size}')  # record this
print(f'n_layer    = {model.config.n_layer}')
print(f'n_embd     = {model.config.n_embd}')
print(f'Parameters = {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M')

In [ ]:
# Cell 5: Verify interleaving and generation controls, then generate a short clip
import os, tempfile, time
from anticipation.sample import generate
from anticipation.convert import midi_to_events, events_to_midi
import anticipation.config as acfg
from model.data.skyline import Note
from model.amt_utils import write_notes_to_midi
from model.training.train_lora import _build_anticipatory_sequence

CO = acfg.CONTROL_OFFSET
melody_notes = [Note(float(t), float(t) + 0.8, p)
                for t, p in enumerate([60, 64, 67, 72])]
tmp = tempfile.NamedTemporaryFile(suffix='.mid', delete=False)
tmp.close()
try:
    write_notes_to_midi(melody_notes, tmp.name)
    events = midi_to_events(tmp.name)
finally:
    os.unlink(tmp.name)
controls = [token + CO for token in events]
interleaved, labels = _build_anticipatory_sequence(events, events, CO)
control_groups = [i for i in range(0, len(interleaved), 3)
                  if interleaved[i + 2] >= CO]
assert len(interleaved) % 3 == 0
assert len(labels) == len(interleaved)
assert control_groups and control_groups[0] == 0
assert any(group > control_groups[0] for group in control_groups), \
    'controls were not interleaved with event groups'
assert any(interleaved[i + 2] < CO for i in range(0, len(interleaved), 3)), \
    'interleaved sequence has no event groups'
print(f'Interleaving check passed: {len(interleaved)//3} groups, '
      f'{len(control_groups)} control groups')

torch.manual_seed(42)
t0 = time.time()
with torch.no_grad():
    output = generate(model, start_time=0, end_time=4, inputs=[],
                      controls=controls, top_p=0.95)
assert len(output) % 3 == 0
print(f'generate() accepted the same control-token format; '
      f'{len(output)//3} output groups in {time.time() - t0:.1f}s')
mid = events_to_midi(output)
mid.save('/content/smoke_test_output.mid')
print('Saved /content/smoke_test_output.mid')

In [ ]:
# Cell 6: 10-step LoRA benchmark
# Reports seconds/step and peak GPU memory.
import torch, time
from peft import LoraConfig, get_peft_model
from torch.cuda.amp import GradScaler, autocast

torch.cuda.reset_peak_memory_stats()

lora_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=['c_attn', 'c_proj', 'c_fc'],
    fan_in_fan_out=True,
    bias='none',
)

# Reload base model fresh for benchmark
try:
    from transformers import AutoModelForCausalLM
    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16).to(device)
except Exception:
    from transformers import GPT2LMHeadModel
    base = GPT2LMHeadModel.from_pretrained(MODEL_NAME, torch_dtype=torch.float16).to(device)

base.enable_input_require_grads()
base.gradient_checkpointing_enable()
lora_model = get_peft_model(base, lora_cfg)
lora_model.print_trainable_parameters()

optimizer = torch.optim.AdamW(
    [p for p in lora_model.parameters() if p.requires_grad], lr=2e-4
)
scaler = GradScaler()

SEQ_LEN = 512
BATCH = 4
vocab = base.config.vocab_size

times = []
for step in range(10):
    ids = torch.randint(0, vocab, (BATCH, SEQ_LEN), device=device)
    t0 = time.time()
    with autocast():
        out = lora_model(input_ids=ids, labels=ids)
        loss = out.loss
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad()
    times.append(time.time() - t0)

peak_gb = torch.cuda.max_memory_allocated() / 1e9
print(f'Mean s/step: {sum(times)/len(times):.2f}')
print(f'Peak GPU memory: {peak_gb:.2f} GB')
print('Smoke test PASSED.')